# 🌱 RAG Híbrido con Reranker para Documentos Agrícolas

Este notebook implementa un sistema de **Retrieval-Augmented Generation (RAG)** con:

- **Búsqueda Léxica** (BM25) para coincidencias exactas de términos
- **Búsqueda Semántica** (Sentence Transformers) para similitud conceptual
- **Fusión Híbrida** (Reciprocal Rank Fusion) para combinar ambas estrategias
- **Reranker** (Cross-Encoder) para reordenar los candidatos por relevancia final

El sistema retorna los **3 chunks más relevantes** dado un prompt de usuario.

---

### Persistencia
Los embeddings generados se pueden **descargar y reutilizar** en ejecuciones futuras para evitar recalcularlos.

## 1. Instalación de Dependencias

In [ ]:
import sys
if 'google.colab' in sys.modules:
    get_ipython().system("pip install -q rank-bm25 sentence-transformers")
else:
    print("Entorno local/VPS. Se asume que las dependencias ya están instaladas.")

## 2. Importaciones

In [ ]:
import json
import os
import pickle
import re
import time
import unicodedata
from typing import Any

import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

# Detectar si estamos en Google Colab
try:
    from google.colab import files as colab_files
    IN_COLAB = True
    print("✅ Ejecutando en Google Colab")
except ImportError:
    IN_COLAB = False
    print("ℹ️ Ejecutando en entorno local")

print("Importaciones completadas.")

## 3. Carga de Datos (`knowledge_base.json`)

Sube el archivo `knowledge_base.json` o colócalo en el directorio de trabajo.

Si ya lo subiste en una ejecución anterior y no reiniciaste el entorno, se detectará automáticamente.

In [ ]:
CHUNKS_FILENAME = "knowledge_base.json"


def load_chunks(filename: str) -> list[dict]:
    """Carga los chunks desde un archivo JSON.

    Si el archivo no existe en el directorio de trabajo y estamos en Colab,
    se solicita al usuario que lo suba manualmente.

    Args:
        filename: Nombre del archivo JSON con los chunks.

    Returns:
        Lista de diccionarios, cada uno representando un chunk con campos
        document_id, citation, date, text, chunk_number.
    """
    if not os.path.exists(filename):
        if IN_COLAB:
            print(f"📂 Sube el archivo '{filename}':")
            uploaded = colab_files.upload()
            if filename not in uploaded:
                raise FileNotFoundError(
                    f"No se subió el archivo esperado '{filename}'. "
                    f"Archivos recibidos: {list(uploaded.keys())}"
                )
        else:
            raise FileNotFoundError(
                f"No se encontró '{filename}' en el directorio actual. "
                f"Colócalo en: {os.getcwd()}"
            )

    with open(filename, "r", encoding="utf-8") as f:
        data = json.load(f)

    chunks = data["chunks"]
    print(f"✅ {len(chunks)} chunks cargados desde '{filename}'.")
    return chunks


chunks = load_chunks(CHUNKS_FILENAME)

# Vista rápida del primer chunk
print(f"\n--- Ejemplo (chunk #1) ---")
print(f"  document_id : {chunks[0]['document_id']}")
print(f"  chunk_number: {chunks[0]['chunk_number']}")
print(f"  text (100c) : {chunks[0]['text'][:100]}...")

## 4. Preprocesamiento para BM25 (Español)

Se implementa un tokenizador simple optimizado para español:
- Conversión a minúsculas
- Remoción de acentos
- Eliminación de puntuación
- Tokenización por espacios

In [ ]:
def preprocess_spanish(text: str) -> list[str]:
    """Preprocesa texto en español para búsqueda BM25.

    Normaliza el texto removiendo acentos, convirtiendo a minúsculas,
    eliminando caracteres no alfanuméricos y tokenizando por espacios.

    Args:
        text: Texto en español a preprocesar.

    Returns:
        Lista de tokens normalizados.
    """
    text = text.lower()
    # Remover acentos: á->a, é->e, ñ se conserva como n
    text = unicodedata.normalize("NFD", text)
    text = "".join(
        c for c in text if unicodedata.category(c) != "Mn"
    )
    # Eliminar puntuación y caracteres especiales
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    # Tokenizar y filtrar tokens vacíos / muy cortos
    tokens = [t for t in text.split() if len(t) > 1]
    return tokens


# Preprocesar todos los chunks para BM25
print("⏳ Preprocesando textos para BM25...")
t0 = time.time()
corpus_tokens = [preprocess_spanish(chunk["text"]) for chunk in chunks]
elapsed = time.time() - t0
print(f"✅ Tokenización completada en {elapsed:.2f}s")
print(f"   Ejemplo tokens chunk #1: {corpus_tokens[0][:10]}...")

# Construir índice BM25
print("\n⏳ Construyendo índice BM25...")
t0 = time.time()
bm25 = BM25Okapi(corpus_tokens)
elapsed = time.time() - t0
print(
    f"✅ Índice BM25 construido en {elapsed:.2f}s ({len(corpus_tokens)} documentos)")

## 5. Embeddings Semánticos con Persistencia

Se usa el modelo multilingüe `paraphrase-multilingual-MiniLM-L12-v2` para generar embeddings densos de cada chunk.

### Persistencia
- Si existe un archivo `embeddings_chunks.pkl` (previamente descargado), se cargará directamente.
- Si no existe, se generan los embeddings y se ofrece **descarga automática**.

In [ ]:
EMBEDDINGS_FILENAME = "embeddings_chunks.pkl"
EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"


def load_or_generate_embeddings(
    chunks: list[dict],
    model_name: str,
    embeddings_file: str,
    batch_size: int = 128,
) -> tuple[np.ndarray, SentenceTransformer]:
    """Carga embeddings desde disco o los genera y ofrece descarga.

    Si el archivo de embeddings existe localmente, lo carga. Si no existe
    y estamos en Colab, solicita la subida. Si aún no hay embeddings,
    los genera con el modelo indicado y ofrece la descarga.

    Args:
        chunks: Lista de chunks con campo 'text'.
        model_name: Nombre del modelo de sentence-transformers.
        embeddings_file: Ruta al archivo .pkl de embeddings.
        batch_size: Tamaño de lote para la codificación.

    Returns:
        Tupla (matriz de embeddings [N, D], modelo cargado).
    """
    model = SentenceTransformer(model_name)
    print(f"✅ Modelo de embeddings cargado: {model_name}")

    # --- Intentar cargar embeddings existentes ---
    embeddings = None

    if os.path.exists(embeddings_file):
        print(f"\n📦 Archivo de embeddings encontrado: '{embeddings_file}'")
        with open(embeddings_file, "rb") as f:
            embeddings = pickle.load(f)
        print(f"✅ Embeddings cargados desde disco: shape={embeddings.shape}")
    elif IN_COLAB:
        print(
            f"\n📂 ¿Tienes embeddings precalculados? Sube '{embeddings_file}' o presiona 'Cancel' para generarlos.")
        try:
            uploaded = colab_files.upload()
            if embeddings_file in uploaded:
                with open(embeddings_file, "rb") as f:
                    embeddings = pickle.load(f)
                print(
                    f"✅ Embeddings cargados desde subida: shape={embeddings.shape}")
        except Exception:
            print("ℹ️ No se subieron embeddings. Se generarán desde cero.")

    # --- Validar dimensiones ---
    if embeddings is not None:
        if embeddings.shape[0] != len(chunks):
            print(
                f"\n⚠️ Dimensión no coincide: embeddings={embeddings.shape[0]}, "
                f"chunks={len(chunks)}. Regenerando..."
            )
            embeddings = None

    # --- Generar si no se cargaron ---
    if embeddings is None:
        texts = [chunk["text"] for chunk in chunks]
        print(
            f"\n⏳ Generando embeddings para {len(texts)} chunks (batch_size={batch_size})...")
        t0 = time.time()
        embeddings = model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        elapsed = time.time() - t0
        print(
            f"✅ Embeddings generados en {elapsed:.1f}s | shape={embeddings.shape}")

        # Guardar en disco
        with open(embeddings_file, "wb") as f:
            pickle.dump(embeddings, f)
        print(f"💾 Embeddings guardados en '{embeddings_file}'")

        # Ofrecer descarga en Colab
        if IN_COLAB:
            print(
                "\n⬇️ Descarga el archivo de embeddings para reutilizarlo en futuras ejecuciones:")
            colab_files.download(embeddings_file)

    return embeddings, model


chunk_embeddings, embedding_model = load_or_generate_embeddings(
    chunks, EMBEDDING_MODEL_NAME, EMBEDDINGS_FILENAME
)

## 6. Pipeline RAG: Búsqueda Híbrida + Reranker

Clase principal que integra:
1. **Búsqueda Léxica** (BM25)
2. **Búsqueda Semántica** (Similitud del coseno con embeddings)
3. **Fusión Híbrida** (Reciprocal Rank Fusion)
4. **Reranker** (Cross-Encoder)

In [ ]:
RERANKER_MODEL_NAME = "BAAI/bge-reranker-base"


class HybridRAGPipeline:
    """Pipeline RAG con búsqueda híbrida (BM25 + Semántica) y reranker.

    Combina los resultados de búsqueda léxica y semántica mediante
    Reciprocal Rank Fusion (RRF), y luego reordena los candidatos
    usando un modelo Cross-Encoder para máxima precisión.

    Attributes:
        chunks: Lista de diccionarios con los chunks del corpus.
        bm25: Índice BM25 pre-construido.
        corpus_tokens: Tokens preprocesados del corpus para BM25.
        chunk_embeddings: Matriz numpy de embeddings del corpus.
        embedding_model: Modelo SentenceTransformer para codificar queries.
        reranker: Modelo CrossEncoder para reordenamiento fino.
    """

    def __init__(
        self,
        chunks: list[dict],
        bm25_index: BM25Okapi,
        corpus_tokens: list[list[str]],
        chunk_embeddings: np.ndarray,
        embedding_model: SentenceTransformer,
        reranker_model_name: str = RERANKER_MODEL_NAME,
    ):
        self.chunks = chunks
        self.bm25 = bm25_index
        self.corpus_tokens = corpus_tokens
        self.chunk_embeddings = chunk_embeddings
        self.embedding_model = embedding_model

        print(f"\n⏳ Cargando modelo reranker: {reranker_model_name}")
        self.reranker = CrossEncoder(reranker_model_name)
        print(f"✅ Reranker cargado: {reranker_model_name}")

    # -------------------------------------------------------------------------
    # Búsqueda Léxica (BM25)
    # -------------------------------------------------------------------------
    def search_bm25(
        self, query: str, top_k: int = 10
    ) -> list[tuple[int, float]]:
        """Busca los top_k chunks más relevantes usando BM25.

        Args:
            query: Pregunta del usuario en texto libre.
            top_k: Número de resultados a retornar.

        Returns:
            Lista de tuplas (indice_chunk, puntaje_bm25) ordenadas por
            relevancia decreciente.
        """
        query_tokens = preprocess_spanish(query)
        scores = self.bm25.get_scores(query_tokens)
        top_indices = np.argsort(scores)[::-1][:top_k]
        return [(int(i), float(scores[i])) for i in top_indices]

    # -------------------------------------------------------------------------
    # Búsqueda Semántica (Cosine Similarity)
    # -------------------------------------------------------------------------
    def search_semantic(
        self, query: str, top_k: int = 10
    ) -> list[tuple[int, float]]:
        """Busca los top_k chunks más similares semánticamente.

        Codifica la query con el modelo de embeddings y calcula la
        similitud del coseno contra todos los embeddings del corpus.

        Args:
            query: Pregunta del usuario en texto libre.
            top_k: Número de resultados a retornar.

        Returns:
            Lista de tuplas (indice_chunk, similitud_coseno) ordenadas
            por similitud decreciente.
        """
        query_embedding = self.embedding_model.encode(
            query, normalize_embeddings=True, convert_to_numpy=True
        )
        # Similitud del coseno (embeddings ya normalizados)
        similarities = self.chunk_embeddings @ query_embedding
        top_indices = np.argsort(similarities)[::-1][:top_k]
        return [(int(i), float(similarities[i])) for i in top_indices]

    # -------------------------------------------------------------------------
    # Fusión Híbrida (Reciprocal Rank Fusion)
    # -------------------------------------------------------------------------
    @staticmethod
    def reciprocal_rank_fusion(
        results_lists: list[list[tuple[int, float]]],
        k: int = 60,
    ) -> list[tuple[int, float]]:
        """Combina múltiples listas de resultados mediante RRF.

        Reciprocal Rank Fusion asigna a cada documento un puntaje basado
        en su posición (rank) en cada lista, usando la fórmula:
            score(d) = sum( 1 / (k + rank_i(d)) ) para cada lista i

        Args:
            results_lists: Lista de listas de resultados. Cada sublista
                contiene tuplas (indice_chunk, puntaje_original).
            k: Parámetro de suavizado (valor estándar: 60).

        Returns:
            Lista de tuplas (indice_chunk, puntaje_rrf) ordenadas por
            puntaje RRF decreciente.
        """
        rrf_scores: dict[int, float] = {}
        for results in results_lists:
            for rank, (idx, _score) in enumerate(results):
                rrf_scores[idx] = rrf_scores.get(
                    idx, 0.0) + 1.0 / (k + rank + 1)
        sorted_results = sorted(
            rrf_scores.items(), key=lambda x: x[1], reverse=True)
        return sorted_results

    # -------------------------------------------------------------------------
    # Reranker (Cross-Encoder)
    # -------------------------------------------------------------------------
    def rerank(
        self,
        query: str,
        candidate_indices: list[int],
    ) -> list[tuple[int, float]]:
        """Reordena los candidatos usando el Cross-Encoder.

        Evalúa cada par (query, texto_candidato) con el modelo reranker
        y retorna los candidatos ordenados por puntaje de relevancia.

        Args:
            query: Pregunta del usuario.
            candidate_indices: Índices de los chunks candidatos.

        Returns:
            Lista de tuplas (indice_chunk, puntaje_reranker) ordenadas
            por relevancia decreciente.
        """
        pairs = [(query, self.chunks[i]["text"]) for i in candidate_indices]
        scores = self.reranker.predict(pairs)
        scored = list(zip(candidate_indices, [float(s) for s in scores]))
        scored.sort(key=lambda x: x[1], reverse=True)
        return scored

    # -------------------------------------------------------------------------
    # Pipeline Completo con Logs
    # -------------------------------------------------------------------------
    def query(
        self,
        user_query: str,
        top_k_lexical: int = 10,
        top_k_semantic: int = 10,
        top_k_final: int = 3,
        verbose: bool = True,
    ) -> list[dict[str, Any]]:
        """Ejecuta el pipeline RAG completo con logs detallados.

        Realiza búsqueda léxica, semántica, fusión RRF y reranking,
        imprimiendo logs detallados de cada etapa si verbose=True.

        Args:
            user_query: Pregunta del usuario en texto libre.
            top_k_lexical: Número de candidatos de BM25.
            top_k_semantic: Número de candidatos semánticos.
            top_k_final: Número de chunks finales a retornar.
            verbose: Si True, imprime logs detallados del proceso.

        Returns:
            Lista de los top_k_final chunks más relevantes, cada uno
            como diccionario con los campos originales más 'rerank_score'.
        """
        separator = "=" * 80
        sub_separator = "-" * 60

        if verbose:
            print(f"\n{separator}")
            print(f"🔍 QUERY: {user_query}")
            print(separator)

        # --- Paso 1: Búsqueda Léxica (BM25) ---
        t0 = time.time()
        bm25_results = self.search_bm25(user_query, top_k=top_k_lexical)
        bm25_time = time.time() - t0

        if verbose:
            print(f"\n📖 PASO 1: BÚsqueda Léxica (BM25) — {bm25_time:.4f}s")
            print(sub_separator)
            print(
                f"  {'Rank':<6} {'Index':<8} {'Doc ID':<25} {'Chunk#':<8} {'BM25 Score':<12} {'Texto (primeros 80 chars)'}")
            print(
                f"  {'----':<6} {'-----':<8} {'------':<25} {'------':<8} {'----------':<12} {'-' * 40}")
            for rank, (idx, score) in enumerate(bm25_results, 1):
                chunk = self.chunks[idx]
                text_preview = chunk['text'][:80].replace('\n', ' ')
                print(
                    f"  {rank:<6} {idx:<8} {chunk['document_id']:<25} {chunk['chunk_number']:<8} {score:<12.4f} {text_preview}...")

        # --- Paso 2: Búsqueda Semántica ---
        t0 = time.time()
        semantic_results = self.search_semantic(
            user_query, top_k=top_k_semantic)
        semantic_time = time.time() - t0

        if verbose:
            print(
                f"\n🧠 PASO 2: BÚsqueda Semántica (Cosine Similarity) — {semantic_time:.4f}s")
            print(sub_separator)
            print(
                f"  {'Rank':<6} {'Index':<8} {'Doc ID':<25} {'Chunk#':<8} {'Cos. Sim.':<12} {'Texto (primeros 80 chars)'}")
            print(
                f"  {'----':<6} {'-----':<8} {'------':<25} {'------':<8} {'---------':<12} {'-' * 40}")
            for rank, (idx, score) in enumerate(semantic_results, 1):
                chunk = self.chunks[idx]
                text_preview = chunk['text'][:80].replace('\n', ' ')
                print(
                    f"  {rank:<6} {idx:<8} {chunk['document_id']:<25} {chunk['chunk_number']:<8} {score:<12.4f} {text_preview}...")

        # --- Paso 3: Fusión Híbrida (RRF) ---
        t0 = time.time()
        hybrid_results = self.reciprocal_rank_fusion(
            [bm25_results, semantic_results]
        )
        fusion_time = time.time() - t0

        if verbose:
            print(
                f"\n🔀 PASO 3: Fusión Híbrida (Reciprocal Rank Fusion) — {fusion_time:.4f}s")
            print(sub_separator)

            # Calcular origen de cada candidato
            bm25_set = {idx for idx, _ in bm25_results}
            semantic_set = {idx for idx, _ in semantic_results}

            print(f"  Candidatos BM25:      {len(bm25_set)}")
            print(f"  Candidatos Semánticos: {len(semantic_set)}")
            print(f"  Intersección:         {len(bm25_set & semantic_set)}")
            print(f"  Candidatos únicos (unión): {len(hybrid_results)}")
            print()
            print(
                f"  {'Rank':<6} {'Index':<8} {'Doc ID':<25} {'Chunk#':<8} {'RRF Score':<12} {'Origen'}")
            print(
                f"  {'----':<6} {'-----':<8} {'------':<25} {'------':<8} {'---------':<12} {'------'}")
            for rank, (idx, rrf_score) in enumerate(hybrid_results, 1):
                chunk = self.chunks[idx]
                in_bm25 = idx in bm25_set
                in_sem = idx in semantic_set
                if in_bm25 and in_sem:
                    origin = "✨ AMBOS"
                elif in_bm25:
                    origin = "📖 BM25"
                else:
                    origin = "🧠 Semántico"
                print(
                    f"  {rank:<6} {idx:<8} {chunk['document_id']:<25} {chunk['chunk_number']:<8} {rrf_score:<12.6f} {origin}")

        # --- Paso 4: Reranking con Cross-Encoder ---
        candidate_indices = [idx for idx, _ in hybrid_results]

        t0 = time.time()
        reranked_results = self.rerank(user_query, candidate_indices)
        rerank_time = time.time() - t0

        if verbose:
            print(
                f"\n🏆 PASO 4: Reranking (Cross-Encoder) — {rerank_time:.4f}s")
            print(sub_separator)
            print(f"  Modelo: {RERANKER_MODEL_NAME}")
            print(f"  Candidatos evaluados: {len(reranked_results)}")
            print()
            print(
                f"  {'Rank':<6} {'Index':<8} {'Doc ID':<25} {'Chunk#':<8} {'Rerank Score':<14} {'Texto (primeros 80 chars)'}")
            print(
                f"  {'----':<6} {'-----':<8} {'------':<25} {'------':<8} {'------------':<14} {'-' * 40}")
            for rank, (idx, score) in enumerate(reranked_results, 1):
                chunk = self.chunks[idx]
                text_preview = chunk['text'][:80].replace('\n', ' ')
                marker = " ⭐" if rank <= top_k_final else ""
                print(
                    f"  {rank:<6} {idx:<8} {chunk['document_id']:<25} {chunk['chunk_number']:<8} {score:<14.6f} {text_preview}...{marker}")

        # --- Resultado Final ---
        final_results = []
        for idx, rerank_score in reranked_results[:top_k_final]:
            result = {**self.chunks[idx],
                      "rerank_score": rerank_score, "_index": idx}
            final_results.append(result)

        total_time = bm25_time + semantic_time + fusion_time + rerank_time

        if verbose:
            print(f"\n{separator}")
            print(
                f"✅ RESULTADO FINAL: Top {top_k_final} chunks más relevantes")
            print(f"⏱️ Tiempo total del pipeline: {total_time:.4f}s")
            print(separator)
            for i, result in enumerate(final_results, 1):
                print(f"\n{'=' * 60}")
                print(f"  🥇 Resultado #{i}")
                print(f"  Rerank Score : {result['rerank_score']:.6f}")
                print(f"  Document ID  : {result['document_id']}")
                print(f"  Chunk Number : {result['chunk_number']}")
                print(f"  Citation     : {result['citation'][:120]}...")
                print(f"  {'~' * 50}")
                print(f"  Texto:")
                # Imprimir texto con indentación
                for line in result['text'].split('\n'):
                    print(f"    {line}")

        return final_results


# --- Instanciar el pipeline ---
print("\n" + "=" * 60)
print("⏳ Inicializando pipeline RAG Híbrido con Reranker...")
print("=" * 60)

rag = HybridRAGPipeline(
    chunks=chunks,
    bm25_index=bm25,
    corpus_tokens=corpus_tokens,
    chunk_embeddings=chunk_embeddings,
    embedding_model=embedding_model,
    reranker_model_name=RERANKER_MODEL_NAME,
)

print("\n✅ Pipeline RAG listo para recibir consultas.")

## 7. 💬 Consulta Interactiva

Ingresa tu pregunta y ajusta los parámetros opcionalmente.

Los logs mostrarán el detalle de cada paso del pipeline:
1. Resultados BM25 (léxicos)
2. Resultados semánticos
3. Fusión híbrida (RRF)
4. Reranking (Cross-Encoder)
5. Resultado final: Top 3 chunks

In [ ]:
# @title 🔍 Consulta al RAG Híbrido
# @markdown ### Ingresa tu pregunta y configura los parámetros:

# @param {type:"string"}
pregunta = "¿Cuales son las condiciones de suelo ideales para el cultivo de cacao?"
top_k_lexical = 10  # @param {type:"slider", min:3, max:30, step:1}
top_k_semantic = 10  # @param {type:"slider", min:3, max:30, step:1}
top_k_final = 3  # @param {type:"slider", min:1, max:10, step:1}

if pregunta.strip():
    results = rag.query(
        user_query=pregunta,
        top_k_lexical=top_k_lexical,
        top_k_semantic=top_k_semantic,
        top_k_final=top_k_final,
        verbose=True,
    )
else:
    print("⚠️ Por favor ingresa una pregunta.")